# TÖL506M Introduction to deep neural networks — Assignment 1
### Given: [date] &nbsp;&nbsp; Due: [date]

**Name:** (your name), **email:** (your email), **collaborators:** (if any)

This assignment covers the material of lectures `00_introduction` and `01_basics`: what
data actually is to a model, what a model is, what an objective function measures, and how
an optimization algorithm changes the parameters. You will meet all four pillars on one
dataset — MNIST — and finish by writing backpropagation yourself before letting PyTorch
write it for you.

Most of the code here is written for you. Your job is to run it, look at what comes out, and
fill in the short gaps marked `____`. Every gap and every question has a **Tip** under it
telling you what you need, so nothing should leave you stuck for long.

Answer every question marked **Answer:** in the markdown cell provided — a couple of
sentences is enough unless the question asks for more. Then send me the finished notebook.
Follow the course rules on collaboration. You may use AI assistance, but disclose what you
used.

Later parts do *not* depend on earlier answers being correct: parts 9–12 only need
`X_train, y_train, X_test, y_test` from part 1. Budget about two hours.

| part | topic | points |
|---|---|---:|
| 1 | Getting the data | 5 |
| 2 | Looking at single digits | 5 |
| 3 | A composite image of all ten digits | 5 |
| 4 | Adding two images together | 5 |
| 5 | The average image of each digit | 5 |
| 6 | PCA per digit: how many directions is a digit? | 10 |
| 7 | Visualising principal components | 10 |
| 8 | A classifier with no learning at all | 5 |
| 9 | One hidden layer, and backpropagation by hand | 20 |
| 10 | Watching the weights learn | 10 |
| 11 | Going deeper | 15 |
| 12 | The same thing in PyTorch | 5 |
| **total** | | **100** |

## Setup

You need `numpy`, `matplotlib`, `scikit-learn`, and (for part 12 only) `torch` and
`torchvision`. If something is missing:

```
pip install numpy matplotlib scikit-learn torch torchvision
```

Everything else in this notebook is written from scratch with NumPy.

### How to work through this notebook

Run the cells in order, from the top. Wherever you see `____`, replace it with a short piece
of code — usually a single expression on one line. The **Tip** underneath each gap tells you
what to use.

If you run a cell without filling a gap you get `NameError: name '____' is not defined`.
That is not a bug in the notebook; it just means you missed one.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
N_SUBSET = 20_000      # how many of the 70000 MNIST images we use; raise it if your machine is fast
TEST_FRACTION = 0.10   # the 10% test set, used for every model in this notebook

rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["image.interpolation"] = "nearest"

## Part 1 — Getting the data — 5 points

MNIST is 70000 handwritten digits, each a $28 \times 28$ grayscale image with a label in
$\{0, \dots, 9\}$. The loader below is given to you. It downloads the dataset once, caches
it in `data/mnist.npz`, and hands back

- `X` of shape $(70000, 784)$, `float32`, every pixel scaled to $[0, 1]$,
- `y` of shape $(70000,)$, `int64`.

Each image has been *flattened*: the $28 \times 28$ grid is laid out row by row into a
vector $\mathbf{x} \in \mathbb{R}^{784}$. Flattening throws away the fact that pixel 0 and
pixel 28 are vertical neighbours — the models in this assignment never learn that they
were ever neighbours. (Convolutional networks, in lecture 07, are the fix.)

In [ ]:
DATA_DIR = Path("data")


def load_mnist():
    """Download MNIST once, cache it, and return X (70000, 784) float32 in [0, 1] and y (70000,) int64."""
    cache = DATA_DIR / "mnist.npz"
    if cache.exists():
        with np.load(cache) as f:
            images, labels = f["images"], f["labels"]
    else:
        DATA_DIR.mkdir(exist_ok=True)
        try:
            from torchvision.datasets import MNIST

            parts = [MNIST(root=str(DATA_DIR), train=t, download=True) for t in (True, False)]
            images = np.concatenate([p.data.numpy() for p in parts]).reshape(-1, 28 * 28)
            labels = np.concatenate([p.targets.numpy() for p in parts])
        except Exception as exc:  # no torchvision, or the mirror is down
            print(f"torchvision route failed ({exc!r}); falling back to OpenML, this is slower")
            from sklearn.datasets import fetch_openml

            bunch = fetch_openml("mnist_784", version=1, as_frame=False)
            images = bunch.data.astype(np.uint8)
            labels = bunch.target.astype(np.int64)
        images = images.astype(np.uint8)
        labels = labels.astype(np.int64)
        np.savez_compressed(cache, images=images, labels=labels)

    return images.astype(np.float32) / 255.0, labels.astype(np.int64)


X_all, y_all = load_mnist()
print(f"X_all {X_all.shape} {X_all.dtype} in [{X_all.min()}, {X_all.max()}]")
print(f"y_all {y_all.shape} {y_all.dtype}, labels {np.unique(y_all)}")

**Your turn.** The shuffling and subsampling is done for you. Fill in the two lines that cut
the shuffled data into a test set (the first `n_test` rows) and a training set (everything
after them).

> **Tip.** `X[:n]` gives you the first `n` rows of an array and `X[n:]` gives you all the
> rest. You need the same cut on both `X` and `y`, so that image `i` keeps its own label.

**This split is used by every later part of the notebook.** Do not resample it.

In [ ]:
order = rng.permutation(len(X_all))[:N_SUBSET]   # shuffle, then keep the first N_SUBSET
X, y = X_all[order], y_all[order]

n_test = int(round(TEST_FRACTION * N_SUBSET))
X_test, y_test = ____, ____        # the first n_test rows of X and of y
X_train, y_train = ____, ____      # everything after the first n_test rows

print(f"X_train {X_train.shape}  y_train {y_train.shape}")
print(f"X_test  {X_test.shape}   y_test  {y_test.shape}")
print("test images per digit:", np.bincount(y_test, minlength=10))

**Question.** What goes wrong if you take the *last* 10% of the unshuffled MNIST array as
your test set instead? Name one concrete way your reported accuracy could be misleading.

> **Tip.** The 70000 images are not in random order: they are the 60000-image training file
> followed by the 10000-image test file, and each of those was written by a different group
> of people. Think about what your test set would then contain, and whether it still looks
> like the data you trained on.

**Answer:** TODO

## Part 2 — Looking at single digits — 5 points

Before any modelling, look at the data. A row vector of 784 numbers becomes an image again
with `.reshape(28, 28)`.

**Your turn.** The figure is set up for you. Fill in the one gap: turn the flat 784-number
row back into a picture that `imshow` can draw.

> **Tip.** `some_row.reshape(28, 28)` folds the 784 numbers back into a 28×28 grid.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(10, 1.7))
for ax, image, label in zip(axes, X_train[:8], y_train[:8]):
    ax.imshow(____, vmin=0, vmax=1)     # `image` is a flat row of 784 numbers
    ax.set_title(int(label))
    ax.axis("off")
fig.suptitle("first eight training images", y=1.18)
plt.show()

**Your turn.** Now look at the raw numbers behind one image, so that "an image is just an
array" from lecture 01 means something concrete. Fill in the gap to print an $8 \times 8$
block from the middle of the picture.

> **Tip.** `image` is a 28×28 array. `image[8:16, 8:16]` takes rows 8 to 15 and columns 8 to
> 15 — the middle of the frame, where the ink usually is.

In [ ]:
image = X_train[0].reshape(28, 28)
print("shape", X_train[0].shape, "dtype", X_train[0].dtype)
print("label", int(y_train[0]))
print(np.round(____, 1))            # an 8x8 block from the middle of `image`

**Question.** The model never sees ink, strokes, or a pen. What exactly is the input
$\mathbf{x}$ to every model in this notebook, and what is the range of its entries?

> **Tip.** Read your own output above: the shape tells you how many numbers there are, and
> the printed block tells you what values they take. Say what one number *means*.

**Answer:** TODO

## Part 3 — A composite image of all ten digits — 5 points

A quick way to sanity-check a labelled image dataset is to build one picture that shows an
example of every class at once. If a label is wrong, or a class is missing, you see it
immediately.

**Your turn.** The loop picks one random training image of each digit and the plotting is
written for you. Fill in the one gap: glue the ten 28×28 tiles side by side into a single
$(28, 280)$ array.

> **Tip.** `np.hstack(list_of_arrays)` stacks arrays horizontally — left to right. (Its
> partner `np.vstack` stacks them top to bottom, which is not what you want here.)

In [ ]:
tiles = []
for digit in range(10):
    candidates = np.flatnonzero(y_train == digit)     # positions of every training image of this digit
    tiles.append(X_train[rng.choice(candidates)].reshape(28, 28))

composite = ____                                      # glue the ten tiles side by side
print("composite shape", composite.shape)

fig, ax = plt.subplots(figsize=(11, 1.5))
ax.imshow(composite, vmin=0, vmax=1)
ax.set_xticks(np.arange(10) * 28 + 14, labels=range(10))
ax.set_yticks([])
ax.set_title("one random training example of each digit")
plt.show()

**Question.** Re-run the cell a few times. Name two ways in which two examples of the
*same* digit differ from each other. These are the variations any classifier has to survive.

> **Tip.** Watch one column — say the 4s — across several re-runs and describe what changes.
> Think about the pen (how thick?), the angle (upright or leaning?), the position in the box,
> and whether people even draw that digit the same shape every time.

**Answer:** TODO

## Part 4 — Adding two images together — 5 points

Images live in $\mathbb{R}^{784}$, and $\mathbb{R}^{784}$ is a vector space, so
$\mathbf{x}_a + \mathbf{x}_b$ is perfectly well defined. The question is whether the result
means anything.

**Your turn.** Two images are picked for you and the plotting is written. Fill in the two
gaps: the sum of the two images, and their average.

> **Tip.** `x_a` and `x_b` are ordinary NumPy arrays of 784 numbers, so `x_a + x_b` adds them
> pixel by pixel and `/ 2` divides every pixel by two. Watch the `[min, max]` in each title —
> that is the point of the exercise.

In [ ]:
x_a = X_train[rng.choice(np.flatnonzero(y_train == 3))]
x_b = X_train[rng.choice(np.flatnonzero(y_train == 8))]

panels = [
    ("a: digit 3", x_a),
    ("b: digit 8", x_b),
    ("a + b", ____),                # the two images added together
    ("(a + b) / 2", ____),          # their average
]

fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, (title, vec) in zip(axes, panels):
    ax.imshow(vec.reshape(28, 28), vmin=vec.min(), vmax=vec.max())
    ax.set_title(f"{title}\n[{vec.min():.1f}, {vec.max():.1f}]", fontsize=9)
    ax.axis("off")
plt.show()

**Question.** Two parts.

1. The sum of two valid images is not a valid image. Give the two distinct reasons visible
   in your plot — one about the pixel *values*, one about the pixel *pattern*.
2. What label would you give the average image? What does that tell you about whether the
   set of "images that look like a handwritten digit" is a linear subspace of
   $\mathbb{R}^{784}$?

> **Tip.** For 1, compare the `[min, max]` in the third title against the first two — real
> images live in $[0, 1]$ — and then just look at the picture and ask whether a pen could
> have drawn it. For 2, try to write a single digit on the answer line for the fourth panel;
> if you cannot, that is the answer, and "linear subspace" means closed under adding and
> scaling.

**Answer:** TODO

## Part 5 — The average image of each digit — 5 points

**Your turn.** Compute the mean training image $\boldsymbol{\mu}_c$ of every digit $c$,

$$\boldsymbol{\mu}_c \;=\; \frac{1}{|\mathcal{I}_c|}\sum_{i \in \mathcal{I}_c} \mathbf{x}_i,
\qquad \mathcal{I}_c = \{\, i : y_i = c \,\},$$

store them in an array `means` of shape $(10, 784)$. The display code is written for you.

> **Tip.** `X_train[y_train == c]` gives you every training image of digit `c` as a block of
> rows. Averaging *down* the rows of that block — `.mean(axis=0)` — leaves one image.
> `np.stack` then puts the ten results into one $(10, 784)$ array.

Keep `means` — part 8 needs it.

In [ ]:
means = np.stack([____ for c in range(10)])      # the mean image of digit c
print("means", means.shape)

fig, axes = plt.subplots(2, 5, figsize=(8, 3.6))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(means[c].reshape(28, 28), vmin=0, vmax=1)
    ax.set_title(c)
    ax.axis("off")
fig.suptitle("mean training image per digit", y=1.02)
plt.tight_layout()
plt.show()

**Question.** The mean images are blurry, and some are blurrier than others. Explain
*mechanically* what averaging does to a pixel that is ink in some examples and background in
others. Which digit gives the sharpest mean image, and why does that fit your explanation?

> **Tip.** Take one pixel. Ink is near 1 and background is 0, so what number do you get if
> 30% of the images have ink there? Then look at your grid and find the digit whose writers
> agree the most about where the stroke goes — it is the one whose shape has the fewest ways
> to be drawn.

**Answer:** TODO

## Part 6 — PCA per digit: how many directions is a digit? — 10 points

Part 5 showed that images of one digit scatter around their mean. Principal component
analysis measures *how* they scatter. For the images of a single digit $c$, PCA finds an
orthonormal basis $\mathbf{v}_1, \mathbf{v}_2, \dots$ of $\mathbb{R}^{784}$ ordered so that
$\mathbf{v}_1$ is the direction of largest variance about $\boldsymbol{\mu}_c$,
$\mathbf{v}_2$ the largest variance orthogonal to it, and so on. The variance along
$\mathbf{v}_j$ is $\lambda_j$, and

$$\text{explained variance ratio}_j = \frac{\lambda_j}{\sum_{m} \lambda_m}.$$

The number of components you need to reach 95% of the variance is a rough measure of how
many degrees of freedom that digit actually has — this is the *dimensionality reduction*
idea from the unsupervised-learning section of lecture 01.

Nothing here uses the labels as a supervision signal; we simply run PCA ten times, once per
digit.

**Your turn.** The loop, the table and the plots are written for you. Fill in two gaps:
which images go into the PCA for digit `c`, and how many components it takes to reach 95%.

> **Tip.** The first gap is the same selection you used in part 5: `X_train[y_train == c]`.
> For the second, `cumulative` is already the running total of explained variance, so you
> need the position where it first reaches 0.95 — `np.searchsorted(cumulative, 0.95) + 1`
> gives exactly that. (The `+ 1` is because positions count from 0 but we are counting
> components, which start at 1.)

In [ ]:
from sklearn.decomposition import PCA

k95 = np.zeros(10, dtype=int)
curves = []

for c in range(10):
    Xc = ____                                            # the training images of digit c
    pca = PCA().fit(Xc)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    k95[c] = ____                                        # how many components reach 95%
    curves.append(cumulative)

print("digit   n_train   components for 95%")
for c in range(10):
    print(f"{c:>5}   {int((y_train == c).sum()):>7}   {k95[c]:>18}")
print(f"\nfewest: digit {int(np.argmin(k95))} ({k95.min()}), "
      f"most: digit {int(np.argmax(k95))} ({k95.max()}), out of 784 pixels")

fig, (ax_curve, ax_bar) = plt.subplots(1, 2, figsize=(11, 3.8))
for c, cumulative in enumerate(curves):
    ax_curve.plot(np.arange(1, len(cumulative) + 1), cumulative, label=str(c), lw=1.2)
ax_curve.axhline(0.95, color="k", ls="--", lw=1)
ax_curve.set_xlim(0, 200)
ax_curve.set_xlabel("number of components")
ax_curve.set_ylabel("cumulative explained variance")
ax_curve.set_title("how fast each digit's variance is captured")
ax_curve.legend(ncol=5, fontsize=8, title="digit")

ax_bar.bar(np.arange(10), k95, color="tab:blue")
ax_bar.set_xticks(range(10))
ax_bar.set_xlabel("digit")
ax_bar.set_ylabel("components for 95%")
ax_bar.set_title("$k_c$ per digit")
for c in range(10):
    ax_bar.text(c, k95[c] + 1, str(k95[c]), ha="center", fontsize=8)
plt.tight_layout()
plt.show()

**Question.** Three parts.

1. Which digit needs the fewest components and which needs the most? Connect this to how
   blurry their mean images were in part 5.
2. Every $k_c$ is far below 784. What does that say about where the digit images sit inside
   $\mathbb{R}^{784}$?
3. PCA is run here without ever using the labels as a target. Which of the three learning
   paradigms from lecture 01 does it belong to?

> **Tip.** For 1, read the two numbers off your own printout, then put its mean image from
> part 5 next to it. For 2, note that 784 is the number of pixels but $k_c$ is well under
> 200 — so the images of one digit do not use most of the room available to them. For 3, the
> three paradigms are supervised, unsupervised and reinforcement learning; ask what `PCA().fit(Xc)`
> was actually given.

**Answer:** TODO

## Part 7 — Visualising principal components — 10 points

A principal component $\mathbf{v}_j$ is itself a vector in $\mathbb{R}^{784}$, so it can be
reshaped to $28 \times 28$ and looked at. It is not an image of a digit: its entries are
positive *and* negative, and it describes a *change* you can add to the mean. Display it
with a diverging colormap centred at zero (`cmap="RdBu_r"`, `vmin=-m`, `vmax=+m`), otherwise
you cannot see which parts are "add ink" and which are "remove ink".

The clearest way to see what a component does is to walk along it. For a chosen digit $c$,

$$\mathbf{x}(\alpha) \;=\; \boldsymbol{\mu}_c \;+\; \alpha \sqrt{\lambda_j}\, \mathbf{v}_j,
\qquad \alpha \in \{-2, -1, 0, 1, 2\},$$

which stays in units of standard deviations along that component, so $\alpha = \pm 2$ is a
plausible-but-extreme member of the class.

**Your turn.** Both figures are written for you. Fill in the two gaps in the second one:
one standard deviation's worth of movement along component $j$, and the image you get by
stepping $\alpha$ of those away from the mean.

`PCA` gives you three things you need: `pca.mean_` is $\boldsymbol{\mu}_c$,
`pca.components_[j]` is $\mathbf{v}_j$, and `pca.explained_variance_[j]` is $\lambda_j$.

> **Tip.** The first gap is $\sqrt{\lambda_j}\,\mathbf{v}_j$, i.e.
> `np.sqrt(pca.explained_variance_[j]) * pca.components_[j]`. The second is then just
> `pca.mean_ + alpha * step`. Change `DIGIT` and re-run to see another digit.

In [ ]:
DIGIT = 3

Xd = X_train[y_train == DIGIT]
pca = PCA(n_components=32).fit(Xd)
print(f"digit {DIGIT}: {len(Xd)} training images")

fig, axes = plt.subplots(1, 9, figsize=(13, 1.9))
axes[0].imshow(pca.mean_.reshape(28, 28), vmin=0, vmax=1)
axes[0].set_title("mean", fontsize=9)
axes[0].axis("off")
for j, ax in enumerate(axes[1:]):
    component = pca.components_[j].reshape(28, 28)
    scale = np.abs(component).max()
    ax.imshow(component, cmap="RdBu_r", vmin=-scale, vmax=scale)
    ax.set_title(f"PC{j + 1}\n{pca.explained_variance_ratio_[j]:.1%}", fontsize=9)
    ax.axis("off")
fig.suptitle(f"mean and first eight principal components of digit {DIGIT}"
             "  (red = add ink, blue = remove ink)", y=1.25)
plt.show()

alphas = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
fig, axes = plt.subplots(4, len(alphas), figsize=(7, 6))
for j in range(4):
    step = ____                                    # one standard deviation along component j
    for ax, alpha in zip(axes[j], alphas):
        ax.imshow(np.clip(____, 0, 1).reshape(28, 28), vmin=0, vmax=1)   # mean, moved by alpha steps
        ax.axis("off")
        if j == 0:
            ax.set_title(f"$\\alpha={alpha:+.0f}$", fontsize=9)
    axes[j, 0].set_ylabel(f"PC{j + 1}")
    axes[j, 0].axis("on")
    axes[j, 0].set_xticks([])
    axes[j, 0].set_yticks([])
fig.suptitle(f"walking along the components of digit {DIGIT}:  "
             r"$\mu + \alpha\sqrt{\lambda_j}\,v_j$", y=0.99)
plt.tight_layout()
plt.show()

**Question.** Describe in words what the first two components of your chosen digit change
as $\alpha$ goes from $-2$ to $+2$. Then answer: is it guaranteed that a principal component
corresponds to something a human can name (slant, thickness, ...)? Why or why not?

> **Tip.** Read the first two rows of the traversal grid left to right and say what is
> changing — thickness? size? lean? position? For the second half, recall that PCA is only
> ever told to capture the most remaining variance in a direction at right angles to the
> earlier ones; nothing in that instruction mentions handwriting.

**Answer:** TODO

## Part 8 — A classifier with no learning at all — 5 points

Here is the simplest possible use of part 5's mean images. Classify a test image by the mean
it is closest to:

$$\hat{y}(\mathbf{x}) \;=\; \operatorname*{arg\,min}_{c \in \{0,\dots,9\}}
\; \lVert \mathbf{x} - \boldsymbol{\mu}_c \rVert_2^2 .$$

In the vocabulary of lecture 01 this is a full **model** (its parameters are the ten vectors
$\boldsymbol{\mu}_c$, 7840 numbers) with an **objective** to report (the error rate) but no
**optimization algorithm** at all: the parameters were computed in closed form by averaging,
not searched for. It is the baseline every later model in this notebook must beat.

Expanding the square,
$\lVert \mathbf{x} - \boldsymbol{\mu}_c \rVert^2
= \lVert \mathbf{x} \rVert^2 - 2\,\mathbf{x}^\top\boldsymbol{\mu}_c + \lVert \boldsymbol{\mu}_c \rVert^2$,
and the first term does not depend on $c$, so a single matrix product
$\mathbf{X}\mathbf{M}^\top$ scores every test image against every mean at once.

**Your turn.** Fill in two gaps: the score of every test image against every mean, and the
accuracy. Everything else — the confusion matrix and the gallery of mistakes — is written
for you.

> **Tip.** For the scores, use the expansion above: `X @ means.T - 0.5 * (means ** 2).sum(axis=1)`.
> That is one matrix of shape `(n_test, 10)`, and the *largest* entry in a row is the
> *closest* mean, which is why the next line uses `np.argmax`.
> For the accuracy, `pred_mean == y_test` gives an array of `True`/`False`, and `.mean()` of
> that is the fraction correct.

In [ ]:
def plot_confusion(y_true, y_pred, title):
    """Given helper: 10x10 confusion matrix, rows = true label, columns = prediction."""
    matrix = np.zeros((10, 10), dtype=int)
    np.add.at(matrix, (y_true, y_pred), 1)

    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(matrix, cmap="Blues")
    for i in range(10):
        for j in range(10):
            if matrix[i, j]:
                ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=7,
                        color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return matrix

In [ ]:
def nearest_mean_predict(X, means):
    '''Return the index of the closest mean for each row of X. No loop over rows.'''
    scores = ____                        # (n, 10): how well each image matches each mean
    return np.argmax(scores, axis=1)


pred_mean = nearest_mean_predict(X_test, means)
acc_mean = ____                          # fraction of test images predicted correctly
print(f"nearest-mean test accuracy: {acc_mean:.2%}   (random guessing: 10.00%)")

matrix = plot_confusion(y_test, pred_mean, f"nearest mean, test accuracy {acc_mean:.2%}")

off_diagonal = matrix - np.diag(np.diag(matrix))
worst = np.dstack(np.unravel_index(np.argsort(off_diagonal, axis=None)[::-1][:3], (10, 10)))[0]
print("most frequent confusions (true -> predicted):",
      ", ".join(f"{t}->{p} ({off_diagonal[t, p]}x)" for t, p in worst))

wrong = np.flatnonzero(pred_mean != y_test)[:8]
fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for ax, i in zip(axes, wrong):
    ax.imshow(X_test[i].reshape(28, 28), vmin=0, vmax=1)
    ax.set_title(f"{y_test[i]} -> {pred_mean[i]}", fontsize=9)
    ax.axis("off")
fig.suptitle("nearest-mean mistakes", y=1.2)
plt.show()

**Question.** Three parts.

1. Report your accuracy. Which pair of digits does this classifier confuse most, and does
   that match the blur you saw in part 5?
2. This classifier compares an image to a *single* prototype per class. Using what part 6
   told you about within-class variation, explain why that is the fundamental limitation.
3. The parameters were never optimised against the error rate. Suppose you were allowed to
   move the ten vectors $\boldsymbol{\mu}_c$ freely to minimise training error — would you
   expect to do better than averaging? Why?

> **Tip.** For 1, the printed line of worst confusions gives you the pairs; go back to your
> part 5 grid and hold those two mean images side by side. For 2, part 6 told you each digit
> spreads over roughly 100 directions, and this model keeps exactly one image per digit — so
> what happens to a 4 that is slanted differently from the average 4? For 3, ask what
> quantity averaging actually minimises, and whether that is the same as the number of test
> images you get right.

**Answer:** TODO

## Part 9 — One hidden layer, and backpropagation by hand — 20 points

Now all four pillars at once. The **model** is a network with one hidden layer of $H$
sigmoid units and a softmax output over the ten classes; the **objective** is cross-entropy;
the **optimization algorithm** is mini-batch stochastic gradient descent; the **data** is the
same split as everywhere else.

### Forward pass

For a mini-batch $\mathbf{X} \in \mathbb{R}^{n \times 784}$ with one-hot targets
$\mathbf{Y} \in \mathbb{R}^{n \times 10}$,

$$
\begin{aligned}
\mathbf{Z}^{(1)} &= \mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}, &
\mathbf{A}^{(1)} &= \sigma\!\left(\mathbf{Z}^{(1)}\right), &
\sigma(z) &= \frac{1}{1 + e^{-z}},\\[4pt]
\mathbf{Z}^{(2)} &= \mathbf{A}^{(1)}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}, &
\hat{\mathbf{Y}} &= \operatorname{softmax}\!\left(\mathbf{Z}^{(2)}\right) \text{ row-wise}, &
L &= -\frac{1}{n}\sum_{i=1}^{n}\sum_{k=0}^{9} Y_{ik}\log \hat{Y}_{ik},
\end{aligned}
$$

with shapes $\mathbf{W}^{(1)} \in \mathbb{R}^{784 \times H}$,
$\mathbf{b}^{(1)} \in \mathbb{R}^{H}$,
$\mathbf{W}^{(2)} \in \mathbb{R}^{H \times 10}$,
$\mathbf{b}^{(2)} \in \mathbb{R}^{10}$. The biases broadcast over the $n$ rows.

### Backward pass

Write $\boldsymbol{\Delta}^{(\ell)} = \partial L / \partial \mathbf{Z}^{(\ell)}$. Because
softmax and cross-entropy are used together, the output error is just the difference between
prediction and target — all the exponentials cancel:

$$
\boldsymbol{\Delta}^{(2)} = \frac{1}{n}\left(\hat{\mathbf{Y}} - \mathbf{Y}\right)
\in \mathbb{R}^{n \times 10}.
$$

From there the chain rule gives, in order,

$$
\begin{aligned}
\nabla_{\mathbf{W}^{(2)}} L &= \mathbf{A}^{(1)\top}\boldsymbol{\Delta}^{(2)}, &
\nabla_{\mathbf{b}^{(2)}} L &= \sum_{i=1}^{n} \boldsymbol{\Delta}^{(2)}_{i,:},\\[4pt]
\boldsymbol{\Delta}^{(1)} &=
\left(\boldsymbol{\Delta}^{(2)}\mathbf{W}^{(2)\top}\right)
\odot \mathbf{A}^{(1)} \odot \left(1 - \mathbf{A}^{(1)}\right), &&\\[4pt]
\nabla_{\mathbf{W}^{(1)}} L &= \mathbf{X}^{\top}\boldsymbol{\Delta}^{(1)}, &
\nabla_{\mathbf{b}^{(1)}} L &= \sum_{i=1}^{n} \boldsymbol{\Delta}^{(1)}_{i,:},
\end{aligned}
$$

where $\odot$ is elementwise multiplication and $\sigma'(z) = \sigma(z)(1 - \sigma(z))$,
which is why $\boldsymbol{\Delta}^{(1)}$ can be written using $\mathbf{A}^{(1)}$ alone. That
factor is worth staring at: $\sigma'$ never exceeds $1/4$, so every layer you pass through
shrinks the gradient by at least a factor of four. Part 11 is where that starts to matter.

Then one SGD step, for every parameter $\theta$ and learning rate $\eta$:
$\theta \leftarrow \theta - \eta \,\nabla_{\theta} L$.

### Given code

`sigmoid`, `softmax`, `one_hot`, `cross_entropy` and `init_params` are written for you —
read them, particularly the two numerical-stability tricks, then implement `forward`,
`backward` and `train`.

In [ ]:
def sigmoid(Z):
    """Stable logistic. exp overflows for very negative z, so branch on the sign."""
    out = np.empty_like(Z, dtype=np.float64)
    positive = Z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-Z[positive]))
    exp_z = np.exp(Z[~positive])
    out[~positive] = exp_z / (1.0 + exp_z)
    return out


def softmax(Z):
    """Row-wise softmax. Subtracting the row max leaves the result unchanged but stops exp from overflowing."""
    shifted = Z - Z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def one_hot(y, n_classes=10):
    """(n,) integer labels -> (n, n_classes) matrix with a single 1 per row."""
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1.0
    return Y


def cross_entropy(probs, Y):
    """Mean cross-entropy of predicted probabilities against one-hot targets."""
    return float(-np.sum(Y * np.log(probs + 1e-12)) / len(Y))


def init_params(n_in, n_hidden, n_out, rng):
    """Xavier-uniform weights, zero biases. The scale keeps the sigmoids off their flat tails at the start."""
    def xavier(fan_in, fan_out):
        limit = np.sqrt(6.0 / (fan_in + fan_out))
        return rng.uniform(-limit, limit, size=(fan_in, fan_out))

    return {
        "W1": xavier(n_in, n_hidden), "b1": np.zeros(n_hidden),
        "W2": xavier(n_hidden, n_out), "b2": np.zeros(n_out),
    }

**Your turn (a).** This is the heart of the assignment, so take it slowly. Each gap is one
line and corresponds to exactly one equation above. The shapes are written next to each gap
— if your shapes come out right, the line is almost certainly right.

> **Tip.** Matrix product is `@`, elementwise product is `*`, and transpose is `.T`.
> Summing a matrix down its rows is `.sum(axis=0)`. Reading the equations off the page:
> $\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}$ becomes `X @ params["W1"] + params["b1"]`,
> and $\mathbf{A}^{(1)\top}\boldsymbol{\Delta}^{(2)}$ becomes `A1.T @ delta2`.
> For `delta1`, remember $\sigma'$ is written using `A1` alone: `A1 * (1.0 - A1)`.
> Do not worry if it is wrong on the first try — the next cell tells you exactly that.

In [ ]:
def forward(params, X):
    Z1 = ____                       # (n, H)    X W1 + b1
    A1 = ____                       # (n, H)    sigmoid of Z1
    Z2 = ____                       # (n, 10)   A1 W2 + b2
    probs = ____                    # (n, 10)   softmax of Z2
    return probs, {"X": X, "A1": A1, "probs": probs}


def backward(params, cache, Y):
    X, A1, probs = cache["X"], cache["A1"], cache["probs"]
    n = len(Y)

    delta2 = ____                   # (n, 10)   (predictions - targets) / n
    grad_W2 = ____                  # (H, 10)
    grad_b2 = ____                  # (10,)     sum delta2 down the rows

    delta1 = ____                   # (n, H)    (delta2 W2^T) * sigma'(Z1)
    grad_W1 = ____                  # (784, H)
    grad_b1 = ____                  # (H,)      sum delta1 down the rows

    return {"W1": grad_W1, "b1": grad_b1, "W2": grad_W2, "b2": grad_b2}

**Check.** Run the cell below. It compares your gradients against finite differences,

$$\frac{\partial L}{\partial \theta_j} \approx
\frac{L(\theta_j + \epsilon) - L(\theta_j - \epsilon)}{2\epsilon},$$

on a tiny random problem — that is, it nudges each parameter by a hair and checks that the
loss moves by the amount your gradient predicted. This is *the* way to debug
backpropagation: if the relative error is above about $10^{-5}$, one of your lines above is
wrong, and no amount of training will fix it.

If it says FAIL, the name printed on the failing line tells you which line to look at: `W2`
or `b2` points at `delta2`/`grad_W2`/`grad_b2`, and `W1` or `b1` points at
`delta1`/`grad_W1`/`grad_b1`. Get a PASS before going on.

In [ ]:
def gradient_check(seed=0, n=7, n_in=6, n_hidden=5, n_out=4, eps=1e-4):
    """Given helper: compare analytic gradients against central finite differences."""
    check_rng = np.random.default_rng(seed)
    params = init_params(n_in, n_hidden, n_out, check_rng)
    X = check_rng.normal(size=(n, n_in))
    Y = one_hot(check_rng.integers(0, n_out, size=n), n_out)

    probs, cache = forward(params, X)
    grads = backward(params, cache, Y)

    worst = 0.0
    for name, value in params.items():
        numeric = np.zeros_like(value)
        for idx in np.ndindex(value.shape):
            original = value[idx]
            value[idx] = original + eps
            plus = cross_entropy(forward(params, X)[0], Y)
            value[idx] = original - eps
            minus = cross_entropy(forward(params, X)[0], Y)
            value[idx] = original
            numeric[idx] = (plus - minus) / (2 * eps)

        scale = np.maximum(np.abs(numeric) + np.abs(grads[name]), 1e-8)
        error = np.max(np.abs(numeric - grads[name]) / scale)
        worst = max(worst, error)
        print(f"{name:>3}  shape {str(value.shape):>10}  max relative error {error:.2e}")

    print("\nPASS" if worst < 1e-5 else "\nFAIL - check your backward pass")
    return worst


_ = gradient_check()

**Your turn (b).** The training loop is written except for the three lines that actually do
the learning: the forward pass, the backward pass, and the update. `accuracy` is given.

> **Tip.** `forward` returns two things, so `probs, cache = forward(params, Xb)`.
> `backward` takes the cache you just made: `backward(params, cache, Yb)`.
> The update is the SGD rule $\theta \leftarrow \theta - \eta\,\nabla_\theta L$, which in
> code is `lr * grads[name]`.

In [ ]:
def accuracy(params, X, y):
    """Given helper: fraction of correct predictions."""
    return float((np.argmax(forward(params, X)[0], axis=1) == y).mean())

In [ ]:
def train(X_tr, y_tr, X_te, y_te, n_hidden=128, lr=1.0, epochs=25,
          batch_size=64, seed=0, verbose=True):
    train_rng = np.random.default_rng(seed)
    params = init_params(X_tr.shape[1], n_hidden, 10, train_rng)
    Y_tr = one_hot(y_tr)
    history = {"train_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        order = train_rng.permutation(len(X_tr))       # a fresh shuffle every epoch
        running_loss = 0.0

        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            Xb, Yb = X_tr[batch], Y_tr[batch]

            probs, cache = ____                        # forward pass on this mini-batch
            running_loss += cross_entropy(probs, Yb) * len(batch)

            grads = ____                               # backward pass on this mini-batch
            for name in params:
                params[name] -= ____                   # one SGD step on this parameter

        history["train_loss"].append(running_loss / len(order))
        history["test_acc"].append(accuracy(params, X_te, y_te))
        if verbose:
            print(f"  epoch {epoch:>2}  train loss {history['train_loss'][-1]:.4f}"
                  f"  test acc {history['test_acc'][-1]:.2%}")

    return params, history

In [ ]:
params_1h, history_1h = train(X_train, y_train, X_test, y_test,
                              n_hidden=128, lr=1.0, epochs=25)
print(f"\none hidden layer of 128 sigmoid units: test accuracy "
      f"{history_1h['test_acc'][-1]:.2%}  (nearest mean was {acc_mean:.2%})")

**Your turn (c).** Now vary the two knobs and watch what happens. The loops and both plots
are written for you; fill in the single call that trains one configuration. This runs six
short trainings and takes about a minute.

> **Tip.** You want `train(X_train, y_train, X_test, y_test, n_hidden=n_hidden, lr=lr,
> epochs=10, verbose=False)`. It returns two things and the line already unpacks them as
> `_, history` — the underscore just means "I do not need this one".

In [ ]:
learning_rates = [0.01, 0.1, 1.0]
hidden_sizes = [32, 128]

sweep = {}
for n_hidden in hidden_sizes:
    for lr in learning_rates:
        _, history = ____                  # train one configuration for 10 epochs, quietly
        sweep[(n_hidden, lr)] = history
        print(f"H={n_hidden:>4}  lr={lr:<5}  final test acc {history['test_acc'][-1]:.2%}")

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
for (n_hidden, lr), history in sweep.items():
    epochs_axis = np.arange(1, len(history["train_loss"]) + 1)
    style = "-" if n_hidden == hidden_sizes[-1] else "--"
    label = f"H={n_hidden}, lr={lr}"
    ax_loss.plot(epochs_axis, history["train_loss"], style, label=label)
    ax_acc.plot(epochs_axis, history["test_acc"], style, label=label)
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("training cross-entropy")
ax_loss.set_title("objective during optimization"); ax_loss.legend(fontsize=8)
ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("test accuracy")
ax_acc.axhline(acc_mean, color="k", ls=":", lw=1)
ax_acc.text(1, acc_mean + 0.01, "nearest mean", fontsize=8)
ax_acc.set_title("test accuracy"); ax_acc.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

**Question.** Four parts.

1. Report the best configuration and its test accuracy. How much better is it than part 8?
2. Describe what $\eta = 0.01$ does to the loss curve and what $\eta = 1.0$ does. Relate
   this to the learning-rate slide in lecture 01.
3. Which knob mattered more here, the learning rate or the hidden width? What does that say
   about where the difficulty in training a network lies?
4. The network has $784H + H + 10H + 10$ parameters. Compute it for $H = 128$ and compare
   with the 7840 parameters of the nearest-mean classifier. Is "more parameters" the whole
   explanation for the improvement? (Look back at part 8, question 3.)

> **Tip.** For 1, read the best line off your printed table. For 2, compare the *shape* of
> the two loss curves — one barely moves, the other drops fast and flattens. For 3, find the
> two runs that disagree most and see which knob they differ in. For 4, put $H = 128$ into
> the formula, then remember from part 8 that a model with only 7850 parameters already gets
> about 91% once it is *trained*.

**Answer:** TODO

## Part 10 — Watching the weights learn — 10 points

So far the network has been a black box that produces a number. It is not: every column of
$\mathbf{W}^{(1)}$ is a vector in $\mathbb{R}^{784}$, exactly like a mean image in part 5 or
a principal component in part 7, so it can be reshaped to $28 \times 28$ and looked at. Hidden
unit $j$ computes $\sigma(\mathbf{x}^\top \mathbf{W}^{(1)}_{:,j} + b_j)$, an inner product
between the image and its weight column — so that column is a *template*, and the unit fires
when the input resembles it.

In this part you snapshot $\mathbf{W}^{(1)}$ during training and watch the templates form out
of random noise. To keep it fast, train on **two digits only**: a couple of hundred steps on
a two-class problem is enough to see the structure appear, where the full ten-class problem
would take long enough to be annoying to re-run while you play with the settings.

**Your turn (a).** Build the two-digit subset. The masks and the relabelling are done for
you; fill in the two gaps that actually select the image rows.

`PAIR = (3, 8)` is a good default — they overlap a lot, so the templates have to work for
their living — but `(0, 1)`, `(4, 9)` and `(5, 3)` are all worth a look.

> **Tip.** `pair_train` is a mask: an array of `True`/`False`, one per training image.
> `X_train[pair_train]` keeps exactly the rows where it is `True`. The label lines just under
> each gap show you the pattern.

In [ ]:
PAIR = (3, 8)

pair_train = np.isin(y_train, PAIR)      # True for every training image whose label is in PAIR
pair_test = np.isin(y_test, PAIR)
relabel = {digit: index for index, digit in enumerate(PAIR)}   # e.g. {3: 0, 8: 1}

X_pair = ____                            # the training images in PAIR
y_pair = np.array([relabel[label] for label in y_train[pair_train]])
X_pair_test = ____                       # the test images in PAIR
y_pair_test = np.array([relabel[label] for label in y_test[pair_test]])

print(f"digits {PAIR}: {len(X_pair)} training, {len(X_pair_test)} test images")
print("training images per class:", np.bincount(y_pair))

**Your turn (b).** `train_with_snapshots` below is given — it is your part 9 loop with one
extra line — so all you have to do is call it. Use `n_hidden=16` (16 templates fit a tidy
$4 \times 4$ grid), `lr=1.0` and `epochs=20`.

> **Tip.** The call is
> `train_with_snapshots(X_pair, y_pair, X_pair_test, y_pair_test, n_classes=2, n_hidden=16, lr=1.0, epochs=20)`.
> It returns four things, and the line already unpacks them for you.

In [ ]:
def train_with_snapshots(X_tr, y_tr, X_te, y_te, n_classes=2, n_hidden=16, lr=1.0,
                         epochs=20, batch_size=64, snapshot_every=20, seed=0):
    """Given: part 9's training loop, plus one line that records a copy of W1."""
    train_rng = np.random.default_rng(seed)
    params = init_params(X_tr.shape[1], n_hidden, n_classes, train_rng)
    Y_tr = one_hot(y_tr, n_classes)

    snapshots, steps = [params["W1"].copy()], [0]
    accuracies = []
    step = 0

    for epoch in range(epochs):
        order = train_rng.permutation(len(X_tr))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            probs, cache = forward(params, X_tr[batch])
            grads = backward(params, cache, Y_tr[batch])
            for name in params:
                params[name] -= lr * grads[name]

            step += 1
            if step % snapshot_every == 0:
                snapshots.append(params["W1"].copy())   # <- the only new line
                steps.append(step)

        accuracies.append(accuracy(params, X_te, y_te))

    return params, np.array(snapshots), np.array(steps), accuracies


def weight_grid(W, n_rows, n_cols, pad=2):
    """Given: lay the first n_rows*n_cols columns of W (784, H) out as one big image.

    Each template is divided by its own maximum absolute value, for the same reason as in
    part (c): one loud unit would otherwise wash out all the others. Tiles are separated by
    a `pad`-pixel gap so that you can tell the 16 units apart.
    """
    tiles = []
    for j in range(n_rows * n_cols):
        tile = W[:, j].reshape(28, 28)
        tile = tile / max(np.abs(tile).max(), 1e-12)
        tiles.append(np.pad(tile, pad, constant_values=0.0))
    return np.vstack([np.hstack(tiles[r * n_cols:(r + 1) * n_cols]) for r in range(n_rows)])

In [ ]:
pair_params, snapshots, snapshot_steps, pair_acc = ____      # train on the two-digit subset

print(f"digits {PAIR}: test accuracy {pair_acc[-1]:.2%} after 20 epochs "
      f"(chance is {max(np.bincount(y_pair_test)) / len(y_pair_test):.2%})")
print(f"{len(snapshots)} snapshots of W1, shape {snapshots.shape[1:]}, "
      f"over {snapshot_steps[-1]} SGD steps")

**Your turn (c).** Now the picture: a **filmstrip** with one row per snapshot time and one
column per hidden unit, so you can read a single template's history down a column. The whole
figure is written for you except for pulling one weight column out of one snapshot.

These are weights, not images: they are positive *and* negative, so the code uses
`cmap="RdBu_r"` with a symmetric range, and scales each panel by its own maximum — otherwise
the random weights at step 0 would be invisible next to the trained ones.

> **Tip.** `snapshots` has shape `(n_snapshots, 784, 16)`. So `snapshots[frame]` is one
> $784 \times 16$ weight matrix, `snapshots[frame][:, unit]` is one column of 784 numbers,
> and you already know how to turn 784 numbers into a picture.

In [ ]:
times = [0, len(snapshots) // 5, len(snapshots) // 2, len(snapshots) - 1]
units = range(8)

fig, axes = plt.subplots(len(times), len(units), figsize=(10, 5.4))
for row, frame in enumerate(times):
    for col, unit in enumerate(units):
        column = ____                       # weight column `unit` of snapshot `frame`, as 28x28
        scale = np.abs(column).max()
        axes[row, col].imshow(column, cmap="RdBu_r", vmin=-scale, vmax=scale)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(f"unit {unit}", fontsize=8)
    axes[row, 0].set_ylabel(f"step {snapshot_steps[frame]}", fontsize=8)
fig.suptitle(f"columns of $W^{{(1)}}$ forming during training on digits {PAIR}"
             "   (red = positive weight, blue = negative)", y=0.99)
plt.tight_layout()
plt.show()

**(d) The reward.** Finally the animation. Both cells below are written for you — just run
them. The helper builds a $4 \times 4$ grid of all 16 templates, writes an animated GIF next
to the notebook, and displays it.

Watch the noise resolve into stroke templates. That is your network learning, one mini-batch
at a time.

In [ ]:
from matplotlib import animation
from PIL import Image as PILImage, ImageSequence
from IPython.display import Image


def animate_weights(snapshots, steps, n_rows=4, n_cols=4,
                    filename="weights_evolution.gif", fps=10, colors=32):
    """Given: animate the weight grid over the snapshots and save it as a GIF."""
    fig, ax = plt.subplots(figsize=(3.2, 3.4), dpi=100)
    ax.axis("off")

    # weight_grid already scales every tile into [-1, 1], so the colour limits are fixed.
    image = ax.imshow(weight_grid(snapshots[0], n_rows, n_cols),
                      cmap="RdBu_r", vmin=-1, vmax=1)
    title = ax.set_title("step 0")

    def update(index):
        image.set_data(weight_grid(snapshots[index], n_rows, n_cols))
        title.set_text(f"step {steps[index]}")
        return image, title

    anim = animation.FuncAnimation(fig, update, frames=len(snapshots), blit=False)
    anim.save(filename, writer=animation.PillowWriter(fps=fps))
    plt.close(fig)

    # Every pixel changes in every frame, so the GIF gets no help from inter-frame
    # compression and comes out large. Cutting the palette to `colors` entries shrinks it
    # several-fold and is invisible on a two-tone colormap.
    frames = [f.convert("RGB").quantize(colors=colors)
              for f in ImageSequence.Iterator(PILImage.open(filename))]
    frames[0].save(filename, save_all=True, append_images=frames[1:],
                   duration=1000 // fps, loop=0, optimize=True)
    return filename

In [ ]:
gif_path = animate_weights(snapshots, snapshot_steps)
print(f"wrote {gif_path}")
Image(gif_path)

**Question.** Three parts.

1. Describe what happens to a single template between step 0 and the end of training. What
   does it look like at the start, and what structure has appeared by the end?
2. The templates are not mean images, even though both are $28 \times 28$ pictures built
   from the training set. Compare a template to the mean images of your two digits from
   part 5: what do the red and the blue regions of a template correspond to, and why does a
   mean image have no negative part at all?
3. Re-run the whole part with `PAIR = (0, 1)`. Do the templates form faster or slower than
   for your first pair, and does the test accuracy agree? Explain the difference in terms of
   what the network has to learn.

> **Tip.** For 1, read one column of your filmstrip from top to bottom. For 2, the key word
> is *sign*: a mean image is an average of pixel brightnesses, which are never negative,
> while a weight can be negative — and a negative weight means "seeing ink here counts
> against this unit". For 3, change `PAIR` at the top of part (a), re-run the four cells, and
> compare; then ask how much of their ink a 0 and a 1 share compared with a 3 and an 8.

**Answer:** TODO

## Part 11 — Going deeper — 15 points

Nothing in part 9's derivation was specific to having exactly one hidden layer. With
$L$ weight matrices and an activation $\phi$, the same two rules run in a loop:

$$
\mathbf{A}^{(0)} = \mathbf{X}, \qquad
\mathbf{A}^{(\ell)} = \phi\!\left(\mathbf{A}^{(\ell-1)}\mathbf{W}^{(\ell)} + \mathbf{b}^{(\ell)}\right)
\ \text{ for } \ell = 1,\dots,L-1, \qquad
\hat{\mathbf{Y}} = \operatorname{softmax}\!\left(\mathbf{A}^{(L-1)}\mathbf{W}^{(L)} + \mathbf{b}^{(L)}\right),
$$

and backwards, starting from $\boldsymbol{\Delta}^{(L)} = (\hat{\mathbf{Y}} - \mathbf{Y})/n$
and walking $\ell = L, L-1, \dots, 1$:

$$
\nabla_{\mathbf{W}^{(\ell)}} L = \mathbf{A}^{(\ell-1)\top}\boldsymbol{\Delta}^{(\ell)},
\qquad
\nabla_{\mathbf{b}^{(\ell)}} L = \sum_i \boldsymbol{\Delta}^{(\ell)}_{i,:},
\qquad
\boldsymbol{\Delta}^{(\ell-1)} =
\left(\boldsymbol{\Delta}^{(\ell)}\mathbf{W}^{(\ell)\top}\right)
\odot \phi'\!\left(\mathbf{Z}^{(\ell-1)}\right).
$$

That is all backpropagation is: cache the activations on the way forward, reuse them on the
way back.

We will need two activations, so the code below takes $\phi$ as an argument. Both of ours
have the convenient property that $\phi'$ can be computed from the layer *output*
$\mathbf{A} = \phi(\mathbf{Z})$ without keeping $\mathbf{Z}$ around:

$$
\sigma'(z) = \sigma(z)\left(1 - \sigma(z)\right) = A(1 - A),
\qquad
\operatorname{relu}(z) = \max(0, z), \quad
\operatorname{relu}'(z) = \mathbb{1}[z > 0] = \mathbb{1}[A > 0].
$$

**Your turn (a).** The network is a list of layers `[{"W": ..., "b": ...}, ...]`. The three
functions are written except for five gaps, and every one of them is a line you already wrote
in part 9 — only now inside a loop.

`init_deep` takes a `scheme`: `"xavier"` is what you used in part 9 (right for sigmoid), and
`"he"` draws $W_{ij} \sim \mathcal{N}(0, 2/\text{fan\_in})$, which is the standard choice for
ReLU. You will need both in part (d).

> **Tip.** For the He gap: `rng.normal(0.0, np.sqrt(2.0 / fan_in), size=(fan_in, fan_out))`.
> In `forward_deep` the gap is one hidden layer's step, `activation(A @ layer["W"] + layer["b"])`
> — the same as part 9's `A1 = sigmoid(X @ W1 + b1)`, just with `A` standing in for whatever
> came before. In `backward_deep` the two gradient gaps are `A_prev.T @ delta` and
> `delta.sum(axis=0)`, exactly as in part 9; the last gap is part 9's `delta1` line with
> `layers[index]["W"]` in place of `W2` and `activation_grad(A_prev)` in place of
> `A1 * (1 - A1)`.

In [ ]:
def relu(Z):
    """Given: the rectified linear unit."""
    return np.maximum(Z, 0.0)


def sigmoid_grad(A):
    """Given: sigma'(z) expressed in terms of A = sigma(z)."""
    return A * (1.0 - A)


def relu_grad(A):
    """Given: relu'(z) expressed in terms of A = relu(z)."""
    return (A > 0.0).astype(A.dtype)

In [ ]:
def init_deep(sizes, rng, scheme="xavier"):
    layers = []
    for fan_in, fan_out in zip(sizes[:-1], sizes[1:]):
        if scheme == "xavier":
            limit = np.sqrt(6.0 / (fan_in + fan_out))
            W = rng.uniform(-limit, limit, size=(fan_in, fan_out))
        elif scheme == "he":
            W = ____                    # normal, mean 0, std sqrt(2 / fan_in), shape (fan_in, fan_out)
        else:
            raise ValueError(f"unknown scheme {scheme!r}")
        layers.append({"W": W, "b": np.zeros(fan_out)})
    return layers


def forward_deep(layers, X, activation=sigmoid):
    activations = [X]
    A = X
    for layer in layers[:-1]:           # every layer except the last one
        A = ____                        # this hidden layer's output
        activations.append(A)
    probs = softmax(A @ layers[-1]["W"] + layers[-1]["b"])
    activations.append(probs)
    return probs, activations


def backward_deep(layers, activations, Y, activation_grad=sigmoid_grad):
    n = len(Y)
    grads = [None] * len(layers)
    delta = (activations[-1] - Y) / n   # the output error, as in part 9

    for index in range(len(layers) - 1, -1, -1):     # walk backwards through the layers
        A_prev = activations[index]
        grads[index] = {"W": ____, "b": ____}
        if index > 0:
            delta = ____                # push the error back through this layer

    return grads

**Check.** Same idea as before, now for arbitrary depth. Just run it: all three lines must
say PASS before you trust anything below.

We check with the sigmoid only, and that is not laziness. Finite differences assume the loss
is smooth in the parameter being nudged, and $\operatorname{relu}$ has a kink at $z = 0$: if
a pre-activation sits within $\epsilon$ of zero, the $+\epsilon$ and $-\epsilon$ evaluations
land on opposite sides of the kink and the "numerical gradient" is meaningless. The check
would then report a failure that says nothing about your code. Since `backward_deep` treats
the activation as a black box, verifying it once with a smooth $\phi$ verifies the loop
itself, which is the part you wrote.

In [ ]:
def gradient_check_deep(sizes, activation=sigmoid, activation_grad=sigmoid_grad,
                        seed=0, n=7, eps=1e-4):
    """Given helper: finite-difference check of forward_deep / backward_deep."""
    check_rng = np.random.default_rng(seed)
    layers = init_deep(sizes, check_rng)
    X = check_rng.normal(size=(n, sizes[0]))
    Y = one_hot(check_rng.integers(0, sizes[-1], size=n), sizes[-1])

    _, activations = forward_deep(layers, X, activation)
    grads = backward_deep(layers, activations, Y, activation_grad)

    worst = 0.0
    for layer, grad in zip(layers, grads):
        for name in ("W", "b"):
            value, numeric = layer[name], np.zeros_like(layer[name])
            for idx in np.ndindex(value.shape):
                original = value[idx]
                value[idx] = original + eps
                plus = cross_entropy(forward_deep(layers, X, activation)[0], Y)
                value[idx] = original - eps
                minus = cross_entropy(forward_deep(layers, X, activation)[0], Y)
                value[idx] = original
                numeric[idx] = (plus - minus) / (2 * eps)
            scale = np.maximum(np.abs(numeric) + np.abs(grad[name]), 1e-8)
            worst = max(worst, np.max(np.abs(numeric - grad[name]) / scale))
    return worst


for depth_sizes in ([6, 5, 4], [6, 5, 5, 4], [6, 5, 5, 5, 4]):
    error = gradient_check_deep(depth_sizes, sigmoid, sigmoid_grad)
    print(f"sigmoid  sizes {str(depth_sizes):<16} max relative error {error:.2e}  "
          f"{'PASS' if error < 1e-5 else 'FAIL'}")

**Given.** `train_deep` is the same optimization loop as part 9, just over the layer
list and with the activation passed through, and `depth_sweep` runs it for depths 1 to 5.
Both are written for you — read them, then run the cell.

In [ ]:
def accuracy_deep(layers, X, y, activation=sigmoid):
    return float((np.argmax(forward_deep(layers, X, activation)[0], axis=1) == y).mean())


def train_deep(X_tr, y_tr, X_te, y_te, hidden_sizes, lr=1.0, epochs=15, batch_size=64,
               activation=sigmoid, activation_grad=sigmoid_grad, scheme="xavier",
               seed=0, verbose=False):
    train_rng = np.random.default_rng(seed)
    layers = init_deep([X_tr.shape[1], *hidden_sizes, 10], train_rng, scheme)
    Y_tr = one_hot(y_tr)
    history = {"train_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        order = train_rng.permutation(len(X_tr))
        running_loss = 0.0

        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            Xb, Yb = X_tr[batch], Y_tr[batch]

            probs, activations = forward_deep(layers, Xb, activation)
            running_loss += cross_entropy(probs, Yb) * len(batch)

            grads = backward_deep(layers, activations, Yb, activation_grad)
            for layer, grad in zip(layers, grads):
                layer["W"] -= lr * grad["W"]
                layer["b"] -= lr * grad["b"]

        history["train_loss"].append(running_loss / len(order))
        history["test_acc"].append(accuracy_deep(layers, X_te, y_te, activation))
        if verbose:
            print(f"  epoch {epoch:>2}  loss {history['train_loss'][-1]:.4f}"
                  f"  test acc {history['test_acc'][-1]:.2%}")

    return layers, history


def depth_sweep(activation, activation_grad, scheme, lr, label):
    results = {}
    print(f"{label}:")
    for depth in range(1, 6):
        layers, history = train_deep(X_train, y_train, X_test, y_test,
                                     hidden_sizes=[64] * depth, lr=lr, epochs=15,
                                     activation=activation, activation_grad=activation_grad,
                                     scheme=scheme)
        n_params = sum(layer["W"].size + layer["b"].size for layer in layers)
        results[depth] = (history, n_params)
        print(f"  {depth} hidden layer(s)  {n_params:>7} parameters  "
              f"final test acc {history['test_acc'][-1]:.2%}")
    return results

**Your turn (b).** Run the sweep for the sigmoid network: depths 1 to 5, 64 units per
hidden layer, $\eta = 1.0$, 15 epochs each. This takes a couple of minutes. The plots are
written for you.

**Report what you actually observe.** The result of this cell is the point of the whole part,
and it is not the result you might expect.

> **Tip.** `depth_sweep(sigmoid, sigmoid_grad, "xavier", 1.0, "sigmoid, Xavier init, lr=1.0")`
> — the arguments are the activation, its derivative, the initialisation scheme, the learning
> rate, and a label for the printout.

In [ ]:
sigmoid_depth = ____                     # depth sweep with sigmoid units and Xavier init

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(11, 4))
for depth, (history, _) in sigmoid_depth.items():
    epochs_axis = np.arange(1, len(history["test_acc"]) + 1)
    ax_acc.plot(epochs_axis, history["test_acc"], label=f"{depth} hidden")
    ax_loss.plot(epochs_axis, history["train_loss"], label=f"{depth} hidden")
ax_acc.axhline(0.1, color="k", ls=":", lw=1)
ax_acc.text(1, 0.12, "chance", fontsize=8)
ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("test accuracy")
ax_acc.set_title("sigmoid: depth vs test accuracy"); ax_acc.legend(fontsize=8)
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("training cross-entropy")
ax_loss.set_title("sigmoid: depth vs training loss"); ax_loss.legend(fontsize=8)
plt.tight_layout()
plt.show()

**(c) The explanation.** That plot needs explaining, and one measurement gives it. The cell
below takes a freshly initialised network at each depth, runs one mini-batch through it, and
compares how big the gradient is at the *first* weight matrix against the *last* one. Just
run it, then read the last column.

Remember that $\sigma' \le 1/4$ everywhere, and that it appears once per layer in the
backward recursion.

In [ ]:
batch = rng.permutation(len(X_train))[:256]
Xb, Yb = X_train[batch], one_hot(y_train[batch])

print("sigmoid network, gradient magnitude at initialisation")
print("depth   mean|dL/dW_first|   mean|dL/dW_last|       ratio")
for depth in range(1, 6):
    fresh = init_deep([784, *([64] * depth), 10], np.random.default_rng(0))
    _, activations = forward_deep(fresh, Xb, sigmoid)
    grads = backward_deep(fresh, activations, Yb, sigmoid_grad)
    first, last = np.abs(grads[0]["W"]).mean(), np.abs(grads[-1]["W"]).mean()
    print(f"{depth:>5}   {first:>15.3e}   {last:>14.3e}   {last / first:>10.1f}x")

**Your turn (d).** Now fix it. The factor that shrinks the gradient is $\phi'$, so change
$\phi$: run exactly the same sweep with ReLU units and He initialisation, at $\eta = 0.1$
(ReLU does not saturate, so it needs a smaller step than the sigmoid did). Everything else —
the comparison plot and the repeat of the (c) measurement — is written for you.

> **Tip.** Same function as in (b), different arguments:
> `depth_sweep(relu, relu_grad, "he", 0.1, "\nrelu, He init, lr=0.1")`.

In [ ]:
relu_depth = ____                        # the same sweep, with relu units and He init

fig, (ax_final, ax_curves) = plt.subplots(1, 2, figsize=(11, 4))
depths = np.arange(1, 6)
ax_final.plot(depths, [sigmoid_depth[d][0]["test_acc"][-1] for d in depths],
              "o-", label="sigmoid + Xavier, lr=1.0")
ax_final.plot(depths, [relu_depth[d][0]["test_acc"][-1] for d in depths],
              "s-", label="relu + He, lr=0.1")
ax_final.axhline(0.1, color="k", ls=":", lw=1)
ax_final.set_xticks(depths)
ax_final.set_xlabel("number of hidden layers"); ax_final.set_ylabel("final test accuracy")
ax_final.set_title("what the activation does to depth"); ax_final.legend(fontsize=8)

for depth, (history, _) in relu_depth.items():
    ax_curves.plot(np.arange(1, len(history["test_acc"]) + 1), history["test_acc"],
                   label=f"{depth} hidden")
ax_curves.set_xlabel("epoch"); ax_curves.set_ylabel("test accuracy")
ax_curves.set_title("relu: depth vs test accuracy"); ax_curves.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\ngradient magnitude at initialisation, depth 5")
print("activation   mean|dL/dW_first|   mean|dL/dW_last|       ratio")
for name, act, act_grad, scheme in (("sigmoid", sigmoid, sigmoid_grad, "xavier"),
                                    ("relu", relu, relu_grad, "he")):
    fresh = init_deep([784, *([64] * 5), 10], np.random.default_rng(0), scheme)
    _, activations = forward_deep(fresh, Xb, act)
    grads = backward_deep(fresh, activations, Yb, act_grad)
    first, last = np.abs(grads[0]["W"]).mean(), np.abs(grads[-1]["W"]).mean()
    print(f"{name:<10}   {first:>15.3e}   {last:>14.3e}   {last / first:>10.1f}x")

best_sigmoid = max(sigmoid_depth[d][0]["test_acc"][-1] for d in depths)
best_relu = max(relu_depth[d][0]["test_acc"][-1] for d in depths)
print(f"\nbest sigmoid network: {best_sigmoid:.2%}   best relu network: {best_relu:.2%}")

**Question.** Four parts.

1. With sigmoid units, does adding hidden layers help? What happens at depth 5, and how does
   the number you get there compare with random guessing?
2. Use your part (c) table to explain (1). Which factor in the backward recursion is
   responsible, and how does the first/last ratio change with depth?
3. What changes when you switch to ReLU, both in the accuracy-vs-depth curve and in the
   gradient ratio at depth 5? Why does $\operatorname{relu}'$ not cause the same problem?
4. The deepest ReLU network still does not beat the 2- or 3-layer one by much. Give one
   reason that has nothing to do with vanishing gradients.

> **Tip.** For 1, read your printed table, and remember that with ten roughly equal classes a
> model that has learned nothing scores about 10%. For 2, look at how the ratio column grows
> as you go down the depths, then count how many times $\sigma'$ appears in the recursion for
> a 5-layer network. For 3, compare the two lines in your left-hand plot, and note that
> $\operatorname{relu}'$ is either 0 or **1** — never $1/4$. For 4, think about the dataset
> and the budget rather than the maths: is MNIST hard enough to need five layers, and did
> every network get enough epochs?

**Answer:** TODO

## Part 12 — The same thing in PyTorch — 5 points

Everything you wrote in parts 9 to 11 exists in PyTorch. `nn.Linear` holds $\mathbf{W}$ and
$\mathbf{b}$; `nn.Sigmoid` is $\sigma$ and `nn.ReLU` is $\operatorname{relu}$;
`nn.CrossEntropyLoss` is softmax plus cross-entropy in one numerically stable object (so the
model outputs *logits* $\mathbf{Z}^{(L)}$, not probabilities); `loss.backward()` is your
entire `backward_deep`, derived automatically by tracking the operations of the forward
pass; and `optimizer.step()` is the parameter update.

**Your turn.** Build the same network as part 9 — one hidden layer of 128 sigmoid units —
and train it the same way. Four gaps, and then you are done. Count how few lines this took
compared with parts 9 and 11.

> **Tip.** `nn.Sequential(nn.Linear(784, 128), nn.Sigmoid(), nn.Linear(128, 10))` is the
> whole model — note it ends with `Linear`, not a softmax, because `nn.CrossEntropyLoss`
> applies the softmax itself. The loss for one batch is `loss_fn(model(Xb_t), yb_t)`. Then
> `loss.backward()` computes every gradient, and `optimizer.step()` applies them — those two
> calls replace everything you wrote in parts 9 and 11.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)

Xtr = torch.from_numpy(X_train).float()
ytr = torch.from_numpy(y_train).long()
Xte = torch.from_numpy(X_test).float()
yte = torch.from_numpy(y_test).long()

model = ____                                    # Linear 784->128, Sigmoid, Linear 128->10
loss_fn = nn.CrossEntropyLoss()                 # softmax + cross-entropy in one object
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

for epoch in range(1, 26):
    for Xb_t, yb_t in loader:
        optimizer.zero_grad()                   # clear last batch's gradients
        loss = ____                             # forward pass and loss on this batch
        ____                                    # backward pass: fills in every .grad
        ____                                    # one SGD step
    with torch.no_grad():
        torch_acc = (model(Xte).argmax(dim=1) == yte).float().mean().item()

print(f"PyTorch, one hidden layer of 128 sigmoid units: {torch_acc:.2%}")
print(f"your NumPy version (part 9):                    {history_1h['test_acc'][-1]:.2%}")
print(f"best NumPy network (part 11):                   {best_relu:.2%}")
print(f"nearest mean (part 8):                          {acc_mean:.2%}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

**Question.** Two parts.

1. Which of the functions you wrote in parts 9 to 11 does PyTorch replace, and which single
   call replaces the largest amount of your code?
2. The two implementations do not give identical accuracy even with the same
   hyperparameters. Give two reasons why not — and say whether the difference is evidence
   that one of them is wrong.

> **Tip.** For 1, go through your own functions one at a time — `sigmoid`, `softmax`,
> `one_hot`, `cross_entropy`, `init_deep`, `forward_deep`, `backward_deep`, the update loop —
> and name the PyTorch object that does each job. For 2, list everything that is random or
> approximate in a training run: where do the starting weights come from, in what order do
> the batches arrive, and how precise are the numbers?

**Answer:** TODO

## What to hand in

Send me this notebook with every cell executed, all plots visible, and every **Answer:**
filled in.

Before you do, run it once from the top with `Kernel -> Restart & Run All` and check that it
gets to the bottom without stopping. A cell that raises an error is a cell I cannot mark, and
`NameError: name '____' is not defined` means you left a gap unfilled somewhere above.

You do not need to send the `data/` folder — it is just the MNIST download, and it rebuilds
itself. Do send `weights_evolution.gif` from part 10 if it did not end up embedded in the
notebook.

**Where this goes next.** Part 4 showed pixel space is not linear; part 6 showed a digit
lives in far fewer than 784 directions; part 8 showed a single prototype per class is not
enough; part 11 showed that stacking sigmoid layers hits the vanishing-gradient wall and
that changing the activation is what gets you past it. Lectures 02–05 pick up exactly there:
automatic differentiation, linear and softmax regression as models in their own right, and
multilayer perceptrons with activations that let depth work.